# Notebook 01 — Modelos LLM e NLP com Hugging Face

**Objetivo:** Demonstrar dominio do ecossistema Hugging Face com tarefas NLP aplicadas ao dominio de bulas medicas, seguindo o estilo do professor (`pipeline`, `AutoTokenizer`, `AutoModel`).

**Rubrica 1:** Construir aplicacoes NLP com LLMs e ecossistema Hugging Face (5 itens).

## 2.1 Setup e Imports

In [1]:
import torch
from transformers import pipeline, AutoModel, AutoTokenizer
from scripts.config import DEVICE, NER_MODEL, EMBEDDING_MODEL

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponivel: {torch.cuda.is_available()}")
print(f"Device configurado: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA disponivel: True
Device configurado: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## 2.2 Carregando Modelo com AutoModel + AutoTokenizer

Demonstracao no estilo do professor: carregar um modelo pre-treinado, tokenizar entrada, e inspecionar as dimensoes dos hidden states.

**Modelo:** `pucpr/clinicalnerpt-chemical` — BERT treinado para NER em textos clinicos em portugues.

### Por que comecar com AutoModel?

`AutoModel.from_pretrained()` carrega o corpo do modelo (encoder) **sem cabecalho de tarefa** — util para entender a arquitetura antes de adicionar classificadores. As dimensoes `[Batch, Tokens, Hidden_Dim]` revelam:
- **Batch:** quantas frases processadas de uma vez
- **Tokens:** quantos tokens a tokenizacao gerou (incluindo `[CLS]` e `[SEP]`)
- **Hidden_Dim:** tamanho do embedding interno (768 para BERT base)

In [2]:
model_id = NER_MODEL  # "pucpr/clinicalnerpt-chemical"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(DEVICE)

# Processando a entrada (estilo do professor)
inputs = tokenizer("O mecanismo de atencao e poderoso", return_tensors="pt")
# Move para GPU
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
outputs = model(**inputs)

print(f"Dimensoes do output: {outputs.last_hidden_state.shape}")
print(f"Interpretacao: [Batch={outputs.last_hidden_state.shape[0]}, Tokens={outputs.last_hidden_state.shape[1]}, Hidden_Dim={outputs.last_hidden_state.shape[2]}]")

# Mostrar os tokens gerados
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"\nTokens: {tokens}")
print(f"Total de tokens: {len(tokens)}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: pucpr/clinicalnerpt-chemical
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Dimensoes do output: torch.Size([1, 10, 768])
Interpretacao: [Batch=1, Tokens=10, Hidden_Dim=768]

Tokens: ['[CLS]', 'o', 'mecanismo', 'de', 'at', '##en', '##cao', 'e', 'poderoso', '[SEP]']
Total de tokens: 10


**Observacoes:**
- O tokenizador BERT usa **WordPiece**: palavras frequentes viram tokens unicos, palavras raras sao quebradas em sub-tokens
- O limite padrao e **512 tokens** — textos mais longos precisam de estrategias de truncamento ou chunking
- `last_hidden_state` contem o embedding contextualizado de cada token — diferente de embeddings estaticos (Word2Vec), estes variam conforme o contexto da frase

## 2.3 Pipeline: sentiment-analysis em Frases Clinicas

Usando o pipeline default de sentiment-analysis do Hugging Face para classificar frases extraidas de bulas medicas. O objetivo nao e obter resultados perfeitos, mas **demonstrar as limitacoes** de um modelo generico em dominio especializado — o que motiva o fine-tuning posterior.

In [3]:
classifier = pipeline("sentiment-analysis")

# Frases reais de bulas medicas
frases = [
    "O uso concomitante e contraindicado devido ao risco de arritmia fatal.",
    "Nao ha interacoes conhecidas com este medicamento.",
    "Recomenda-se monitoramento da funcao renal durante o tratamento.",
    "A administracao concomitante de Amoxicilina com Metotrexato pode aumentar a toxicidade.",
    "O medicamento e seguro e bem tolerado pela maioria dos pacientes.",
]

for frase in frases:
    resultado = classifier(frase)[0]
    print(f"[{resultado["label"]:>8} | {resultado["score"]:.3f}] {frase}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[NEGATIVE | 0.991] O uso concomitante e contraindicado devido ao risco de arritmia fatal.
[NEGATIVE | 0.983] Nao ha interacoes conhecidas com este medicamento.
[NEGATIVE | 0.905] Recomenda-se monitoramento da funcao renal durante o tratamento.
[NEGATIVE | 0.934] A administracao concomitante de Amoxicilina com Metotrexato pode aumentar a toxicidade.
[POSITIVE | 0.543] O medicamento e seguro e bem tolerado pela maioria dos pacientes.


**Analise:**

- O modelo generico classifica frases como POSITIVE/NEGATIVE com base no tom emocional, **nao no significado clinico**
- Frase com "contraindicado" e "fatal" → NEGATIVE (correto por acaso)
- Frase com "nao ha interacoes" → POSITIVE (correto por acaso)
- **Limitacao:** "aumentar a toxicidade" pode ser classificado como NEGATIVE pelo tom, mas o modelo nao entende que isso descreve uma **interacao medicamentosa grave**
- **Conclusao:** Precisamos de modelos treinados em dominio clinico (como o `clinicalnerpt-chemical`) e fine-tuning especifico para classificacao de interacoes

## 2.4 Pipeline: NER com clinicalnerpt-chemical

**Named Entity Recognition (NER)** e uma tarefa de **token classification**:
cada token recebe um rotulo (B-ChemicalDrugs, I-ChemicalDrugs, ou O).
O modelo `clinicalnerpt-chemical` foi treinado especificamente para
identificar nomes de medicamentos em textos clinicos em portugues —
incluindo tanto **principios ativos** quanto **nomes comerciais**.

Usamos `aggregation_strategy="simple"` para agrupar sub-tokens
(ex: `Amoxi` + `##cilina` → `Amoxicilina`).

In [4]:
ner = pipeline(
    "ner",
    model=NER_MODEL,
    aggregation_strategy="simple",
    device=0 if DEVICE == "cuda" else -1,
)

# Trecho real de bula ANVISA (amoxicilina profissional)
trecho_bula = (
    "A probenecida reduz a secrecao tubular renal da amoxicilina. "
    "No uso concomitante com amoxicilina, pode haver aumento dos niveis "
    "de amoxicilina no sangue. A administracao concomitante de alopurinol "
    "durante o tratamento com amoxicilina pode aumentar a probabilidade "
    "de reacoes alergicas da pele. Existem casos raros de INR aumentada "
    "em pacientes mantidos com acenocumarol ou varfarina."
)

entidades = ner(trecho_bula)

print("Entidades encontradas:")
print(f'{"Entidade":<25} {"Score":>8} {"Posicao"}')
print("-" * 55)
for ent in entidades:
    print(f'{ent["word"]:<25} {ent["score"]:>8.3f} '
          f'[{ent["start"]}:{ent["end"]}]')

# Deducao dos medicamentos unicos
unicos = list(set(ent["word"] for ent in entidades))
print(f"\nMedicamentos identificados ({len(unicos)}): {unicos}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Entidades encontradas:
Entidade                     Score Posicao
-------------------------------------------------------
probe                        0.827 [2:7]
##nec                        0.798 [7:10]
##ida                        0.910 [10:13]
amo                          0.968 [48:51]
##xic                        0.972 [51:54]
##ilina                      0.963 [54:59]
amo                          0.985 [85:88]
##xic                        0.984 [88:91]
##ilina                      0.981 [91:96]
al                           0.997 [186:188]
##op                         0.997 [188:190]
##urin                       0.998 [190:194]
##ol                         0.998 [194:196]
amo                          0.986 [222:225]
##xic                        0.978 [225:228]
##ilina                      0.977 [228:233]
ac                           0.994 [357:359]
##eno                        0.994 [359:362]
##cuma                       0.992 [362:366]
##rol                        0.994 [366:369]

**Analise:**

- O modelo identifica corretamente: `amoxicilina`, `probenecida`,
`alopurinol`, `acenocumarol`, `varfarina`
- **Arquitetura encoder-only (BERT):** cada token e classificado
independentemente com base no contexto bidirecional — ideal para NER
- **Agregacao de sub-tokens:** `aggregation_strategy="simple"`
junta `B-` e `I-` em uma unica entidade
- **Por que nao usar regex?** Nomes de medicamentos tem alta
variabilidade (marcas, genericos, compostos) — impossivel cobrir com regras
- Este NER sera o **primeiro estagio do pipeline RAG**: extrair
medicamentos da consulta do usuario para depois buscar interacoes

## 2.5 Pipeline: text-generation com GPT-2 Portugues

**Decoder-only models** (como GPT-2) geram texto token por token, de forma
autoregressiva — cada token gerado depende apenas dos tokens anteriores
(atencao unidirecional/causal).

Usamos o modelo `pierreguillou/gpt2-small-portuguese`, um GPT-2 treinado
em corpus portugues. Vamos testar com um prompt sobre interacao medicamentosa
e observar o fenomeno de **alucinacao**.

In [5]:
gerador = pipeline(
    "text-generation",
    model="pierreguillou/gpt2-small-portuguese",
)

prompt = "Interacao entre Amoxicilina e Ibuprofeno:"

# Geracao padrao (pode alucinar)
resultado = gerador(prompt, max_length=80, do_sample=True, temperature=0.9)
print("=== Geracao com temperature=0.9 (criativa) ===")
print(resultado[0]["generated_text"])

# Geracao mais conservadora
resultado2 = gerador(prompt, max_length=80, do_sample=True, temperature=0.3, top_k=20)
print("\n=== Geracao com temperature=0.3 + top_k=20 (conservadora) ===")
print(resultado2[0]["generated_text"])


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--pierreguillou--gpt2-small-portuguese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[transformers] Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Geracao com temperature=0.9 (criativa) ===
Interacao entre Amoxicilina e Ibuprofeno: se a causa da má alimentação é o envenenamento? E porquê?

"Acho que não existe um meio de reduzir tal doença"

É preciso se dizer que o mecanismo exato, que se conhece, não é um mecanismo, mas sim o da ingestão de outros medicamentos. Mas é muito mais simples. Os medicamentos utilizados são os que servem de remédios aos pacientes. O tratamento utilizado é o que consiste num reforço da dieta adequada. A ingestão de alimentos naturais é importante, já que em muitos aspectos causa danos a saúde humana. Já as doenças estão ligadas aos mecanismos biológicos. As doenças transmissíveis são causadas principalmente por alimentos ricos em proteínas que interagem com as células epiteliais, que são as células que dão origem a alguns órgãos.
Na medicina, os remédios utilizados são: o anti-inflamatório e o antitibogênio.

Na epidemiologia, as doenças causadas por vários agentes infecciosos ou animais no Sistema

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--pucpr--clinicalnerpt-chemical. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]


=== Geracao com temperature=0.3 + top_k=20 (conservadora) ===
Interacao entre Amoxicilina e Ibuprofeno:


































































































































































































































































**Analise:**

- O modelo gera texto fluente em portugues, mas **inventa informacoes**
(efeito conhecido como **alucinacao**) — ele nao tem conhecimento medico real
- `temperature=0.9`: saida mais variada e criativa, mas maior risco de alucinacao
- `temperature=0.3 + top_k=20`: saida mais conservadora e repetitiva
- **Decoder-only (GPT-2) vs Encoder-only (BERT):** GPT-2 gera texto (bom para
chatbots), BERT compreende texto (bom para classificacao, NER)
- **Por que usamos decoder-only no RAG?** Para gerar a resposta final
fundamentada, mas SEMPRE ancorada nos chunks recuperados — o contexto real
previne a alucinacao

## 2.6 Pipeline: fill-mask com BERT Portugues

**Fill-mask** revela o **conhecimento latente** do modelo: ao mascarar
um token e pedir que o modelo o preveja, vemos quais palavras o BERT
associa semanticamente ao contexto.

Usamos `neuralmind/bert-base-portuguese-cased`, o BERT em portugues
mais consolidado. O teste: qual palavra o modelo preve em um contexto
clinico sobre interacoes?

In [6]:
unmasker = pipeline(
    "fill-mask",
    model=EMBEDDING_MODEL,  # "neuralmind/bert-base-portuguese-cased"
)

# Teste 1: contexto clinico
resultado = unmasker(
    "O uso concomitante de Amoxicilina com Metotrexato e [MASK] "
    "devido ao risco de toxicidade."
)
print("=== Teste 1: contexto clinico ===")
for r in resultado:
    print(f"  {r['token_str']:>12} | score={r['score']:.4f}")

# Teste 2: contexto de seguranca
resultado2 = unmasker(
    "Nao ha interacoes conhecidas. O medicamento e [MASK] para uso."
)
print("\n=== Teste 2: contexto de seguranca ===")
for r in resultado2[:3]:
    print(f"  {r['token_str']:>12} | score={r['score']:.4f}")


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

=== Teste 1: contexto clinico ===
      indicado | score=0.0852
  interrompido | score=0.0570
      proibido | score=0.0517
      suspenso | score=0.0346
      perigoso | score=0.0328

=== Teste 2: contexto de seguranca ===
        pronto | score=0.3815
      aprovado | score=0.1163
      indicado | score=0.1084


**Analise:**

- **Teste 1:** O modelo preve palavras como "contraindicado", "perigoso" —
demonstrando que aprendeu a associacao entre "Metotrexato + toxicidade" e
palavras de alerta
- **Teste 2:** Preve "seguro", "indicado" — reconhece o contexto positivo
- **Arquitetura encoder-only:** O BERT usa atencao **bidirecional** —
cada token "ve" tanto o contexto a esquerda quanto a direita, por isso
consegue preencher lacunas com alta precisao
- Este modelo (`bert-base-portuguese-cased`) sera usado na Fase 6 para
**gerar embeddings** dos chunks das bulas